# S4.5a — fit `w` as a sensitivity curve (Kaggle)

**RUNNER ONLY.** This notebook contains no experiment logic. It clones the canonical repo, installs the artifact-compatible environment, calls the read-only preflight, calls `src/eval/fit_w.py`, and packages its outputs.

It does **not** generate text, rebuild either trained artifact, touch Gold-300, or select one value of `w`. The complete curve is the registered deliverable.

Kaggle settings: Internet ON; GPU is recommended for LaBSE scoring. No model or dataset input is required because both generation archives and both fitted artifacts are committed. This runner is batch-safe: **Save Version → Save & Run All** may run unattended because every post-install check and experiment command starts a fresh Python subprocess.


## 1. Clean, single checkout

This intentionally uses `/kaggle/working/wfit_repo`, never `/repo/repo`. A separate path prevents an older generation checkout and the current scoring checkout from being mixed.


In [ ]:
from pathlib import Path
import os, subprocess
REPO = Path('/kaggle/working/wfit_repo')
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/alphapie77/BSc_Thesis.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['git', 'log', '--oneline', '-1'], check=True)
for path in ['configs/s4_w.yaml', 'src/eval/preflight_w.py', 'src/eval/fit_w.py', 'results/s4_devplot_generations.jsonl', 'results/s4_devplot_lenctl_generations.jsonl']:
    assert Path(path).is_file(), f'missing from checkout: {path}'


## 2. Artifact-compatible environment

The symbolic scorer was persisted under scikit-learn 1.9.0. The failed notebook used Kaggle's 1.6.1, which warned during unpickling and crashed later in `predict_proba`. These versions come from the committed lock/snapshot, not from a new tuning choice.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'scikit-learn==1.9.0', 'sentence-transformers==5.6.1', 'transformers==5.14.1', 'pyyaml', 'joblib'], check=True)
print('INSTALL COMPLETE. No manual restart is needed: every remaining command runs in a fresh subprocess.')


In [ ]:
import subprocess, sys
gate = "import sklearn, sentence_transformers, transformers; print('scikit-learn       :', sklearn.__version__); print('sentence-transformers:', sentence_transformers.__version__); print('transformers       :', transformers.__version__); assert sklearn.__version__ == '1.9.0'"
subprocess.run([sys.executable, '-c', gate], check=True)


## 3. Read-only preflight

This is the real gate: both registered archives must contain 120 unique generations, and both Critic components must successfully produce a probability. A non-zero exit stops the cell.


In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/kaggle/working/wfit_repo')
os.chdir(REPO)
subprocess.run([sys.executable, 'src/eval/preflight_w.py', '--config', 'configs/s4_w.yaml'], check=True)


## 4. Fit the registered curve

No single `w` is selected. The script scores both archives and writes the full sensitivity curve plus the held-out, plot-grouped marginal-value test.


In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/kaggle/working/wfit_repo')
os.chdir(REPO)
subprocess.run([sys.executable, 'src/eval/fit_w.py', '--config', 'configs/s4_w.yaml'], check=True)


## 5. Save before the Kaggle session disappears

The environment snapshot is written with `--out`, so this external host never rewrites the committed lock file. Download all five files into the repo's `results/` directory; interpretation and step closure happen only after they are on disk locally.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
REPO = Path('/kaggle/working/wfit_repo')
os.chdir(REPO)
snapshot = 'results/env_snapshot_s4w_kaggle.json'
subprocess.run([sys.executable, 'src/common/env_snapshot.py', '--out', snapshot], check=True)
files = ['s4_w_sensitivity.md', 's4_w_sensitivity.json', 's4_w_sensitivity.csv', 's4_w_scores.csv', 'env_snapshot_s4w_kaggle.json']
for name in files:
    src = Path('results') / name
    assert src.is_file(), f'missing output: {src}'
    shutil.copy2(src, Path('/kaggle/working') / name)
    print('saved', name, src.stat().st_size, 'bytes')
